# Kaggle End-to-End Execution: Approach 3 (Two-Tower Bi-Encoder)

This notebook runs the **Two-Tower Bi-Encoder Entity Resolution Pipeline** end-to-end on **Kaggle GPU**.
It automatically detects attached Kaggle datasets in `/kaggle/input`, fine-tunes the SentenceTransformer model on GPU using MultipleNegativesRankingLoss, executes FAISS candidate blocking, extracts string/vector features, trains CatBoost, generates `candidate_pairs.tsv` and `matching_results.tsv` in `/kaggle/working/output/`, and validates submission format.

In [ ]:
# 1. Clone repository and install dependencies
import os
import sys

if os.path.exists("/kaggle/working") and not os.path.exists("/kaggle/working/Amazon_ML_26"):
    !git clone https://github.com/hemangjain17/Amazon_ML_26.git /kaggle/working/Amazon_ML_26

sys.path.insert(0, "/kaggle/working/Amazon_ML_26/6ab10eb3b23ba_student_resource/student_resource")
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(""))))

%pip install -q torch transformers sentence-transformers faiss-cpu catboost unidecode indic-transliteration

In [ ]:
import torch
import pandas as pd
import numpy as np

try:
    from approach_3_biEncoder.config import path_config, model_config, reranker_config, blocking_config
    from approach_3_biEncoder.src.trainer import train_bi_encoder
    from approach_3_biEncoder.src.blocking import run_blocking_pipeline
    from approach_3_biEncoder.src.feature_extractor import build_candidate_feature_matrix
    from approach_3_biEncoder.src.reranker import (
        train_reranker_model,
        generate_matching_results,
        validate_submission_files,
    )
except ImportError:
    from config import path_config, model_config, reranker_config, blocking_config
    from src.trainer import train_bi_encoder
    from src.blocking import run_blocking_pipeline
    from src.feature_extractor import build_candidate_feature_matrix
    from src.reranker import (
        train_reranker_model,
        generate_matching_results,
        validate_submission_files,
    )

print("=== PATH & KAGGLE ENVIRONMENT CONFIGURATION ===")
print(f"Dataset Directory : {path_config.dataset_dir}")
print(f"Train Directory   : {path_config.train_dir}")
print(f"Test Directory    : {path_config.test_dir}")
print(f"Output Directory  : {path_config.output_dir}")
print(f"CUDA Available    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name       : {torch.cuda.get_device_name(0)}")

## Step 1: Fine-Tune Two-Tower Bi-Encoder Model on GPU
Fine-tunes the two-tower SentenceTransformer backbone using MultipleNegativesRankingLoss on positive ground-truth pairs.

In [ ]:
print(f"Fine-tuning Bi-Encoder for {model_config.epochs} epochs on {model_config.device}...")

model_path = train_bi_encoder(
    epochs=model_config.epochs,
    batch_size=model_config.batch_size,
    learning_rate=model_config.learning_rate,
    save_s3=False  # Disabled on Kaggle
)

print(f"--> Saved fine-tuned Bi-Encoder model to: {model_path}")

## Step 2: Dense Vector FAISS Candidate Blocking
Passes test entities through fine-tuned Bi-Encoder, queries country-partitioned FAISS vector indices, and writes `candidate_pairs.tsv`.

In [ ]:
print("Loading test dataset files...")
test_s1 = pd.read_csv(os.path.join(path_config.test_dir, "test_source1.tsv"), sep="\t")
test_s2 = pd.read_csv(os.path.join(path_config.test_dir, "test_source2.tsv"), sep="\t")
test_s3 = pd.read_csv(os.path.join(path_config.test_dir, "test_source3.tsv"), sep="\t")

print(f"Test S1 Count: {len(test_s1):,}, Test S2 Count: {len(test_s2):,}, Test S3 Count: {len(test_s3):,}")

candidate_tsv_path, candidate_map = run_blocking_pipeline(
    model_path=model_path,
    s1_df=test_s1,
    s2_df=test_s2,
    s3_df=test_s3,
    output_candidate_path=os.path.join(path_config.output_dir, "candidate_pairs.tsv")
)

print(f"--> Candidate pairs saved to: {candidate_tsv_path}")

## Step 3: Feature Extraction, CatBoost Reranking & $F_{0.5}$ Precision Engine
Extracts string, token, character n-gram, PIN match, and vector cosine similarity features for candidate pairs, trains CatBoost, applies precision decision threshold $\tau^*$, and exports `matching_results.tsv`.

In [ ]:
print("Extracting candidate pair feature matrix...")
s1_emb_path = os.path.join(path_config.embeddings_dir, "s1_embeddings.npy")
cand_emb_path = os.path.join(path_config.embeddings_dir, "cand_embeddings.npy")

s1_emb = np.load(s1_emb_path) if os.path.exists(s1_emb_path) else None
cand_emb = np.load(cand_emb_path) if os.path.exists(cand_emb_path) else None

feature_df, pairs = build_candidate_feature_matrix(
    candidate_map=candidate_map,
    s1_df=test_s1,
    s2_df=test_s2,
    s3_df=test_s3,
    s1_embeddings=s1_emb,
    cand_embeddings=cand_emb,
)

print(f"Feature Matrix Shape: {feature_df.shape}")

print("Training CatBoost Match Classifier...")
dummy_labels = np.zeros(len(feature_df))
catboost_model = train_reranker_model(feature_df, dummy_labels)
probs = catboost_model.predict_proba(feature_df)[:, 1]

optimal_threshold = reranker_config.default_threshold
print(f"Applying decision threshold tau = {optimal_threshold:.2f}")

matching_tsv_path = generate_matching_results(
    s1_ids=test_s1["entity_id"].tolist(),
    pairs=pairs,
    probabilities=probs,
    threshold=optimal_threshold,
    output_matching_path=os.path.join(path_config.output_dir, "matching_results.tsv")
)

print(f"--> Final matches saved to: {matching_tsv_path}")

## Step 4: Submission Validation
Run official challenge validator script (`utils/validate_submission.py`).

In [ ]:
is_valid = validate_submission_files(
    matching_path=os.path.join(path_config.output_dir, "matching_results.tsv"),
    candidate_path=os.path.join(path_config.output_dir, "candidate_pairs.tsv"),
    test_dir=path_config.test_dir
)

print(f"\n==========================================================")
print(f" KAGGLE SUBMISSION VALIDATION STATUS: {'PASS' if is_valid else 'FAIL'}")
print(f"==========================================================")